# Project — Spotting Gluten Grains from Protein Sequence

Wheat, barley, and rye trigger celiac disease. Rice, maize, oats, quinoa, millet, and
buckwheat do not. In this project you'll test whether you can tell the two apart **from
protein sequence alone**, using only the tools from this course.


## What you'll do

The project has six parts. Each one ends with something concrete you can check.

| Part | You will... | New skill |
|---|---|---|
| **0. Orientation** | Load the data and learn what one row means | reading a data table |
| **1. Sequence to numbers** | Write functions that turn a protein string into numbers (how much of it is proline, how much is glutamine) | functions, `apply` |
| **2. Exploration** | Compare gluten vs non-gluten grains with a histogram and a scatter plot; find which "safe" grains look suspicious | grouped visualization |
| **3. Is the difference real?** | Run a permutation test on the gap between the two groups | hypothesis testing |
| **4. How big is it?** | Put a confidence interval on that gap | the bootstrap |
| **5. A sharper feature** | Count a *repeating pattern* instead of raw amounts, and watch the borderline grains separate | more string logic |
| **6. Prediction** | Build a k-nearest-neighbors classifier, test it honestly on held-out data, and see where it fails | classification |

This preview contains **Part 0** and **Part 1**.


## Why this project

**The real question.** Some grains harm people with celiac disease because of a protein
called **gluten**. Gluten is hard to digest, and fragments of it set off an immune reaction
in the small intestine. Researchers study exactly how those fragments fit into an immune
receptor using 3-D structure prediction. This project asks a simpler version of the same
question that fits in an intro course: *can the amino-acid makeup of a grain's proteins,
by itself, separate the harmful grains from the harmless ones?* And there's a built-in
answer key — we already know which grains are which — so you can tell whether your method
works.

**This is bioinformatics.** Bioinformatics (also called computational biology) means
answering biological questions with code and statistics applied to large datasets —
DNA, protein sequences, clinical measurements. The core skills are the ones this course
teaches: load and clean data, visualize it, test whether a pattern is real, and build a
model that predicts. The extra ingredient is a little biology vocabulary, which this
notebook gives you as you go.

**As a career.** Roles that combine data skills with biology have grown alongside the size
of biological datasets — in biotechnology and pharmaceutical companies, hospital and
public-health laboratories, agricultural research, and academic labs. You do not need a
biology degree to enter the field; a data-science foundation plus domain curiosity is a
common path. This project is a small proof of that: you'll reach a real result with nothing
beyond this course.


## The biology you need (all of it)

* A **protein** is a molecule built from a chain of small parts called **amino acids**.
  There are 20 kinds. Each is written as one capital letter:
  `A C D E F G H I K L M N P Q R S T V W Y`.
* So a protein can be written as a **string of letters**. For example, one real protein in
  our data begins:

  ```
  MAAKIFALLALHALLVSGTTAAIIPQCSLAPNAIIPQFIPPVTALGNEHLAVQAIRLQQ...
  ```

  That is not a misspelled word. It is a chain of 267 amino acids, one letter each, read
  left to right.
* Two of the letters matter here: **`P`** (the amino acid *proline*) and **`Q`**
  (*glutamine*). The storage protein in wheat, barley, and rye is unusually rich in `P`
  and `Q`, and that is chemically why it triggers celiac disease.
* In our data, each row is one protein. The `sequence` column holds its string. The
  `gluten` column says whether that protein's grain is one of the three that cause celiac
  disease. **Predicting `gluten` from `sequence` is the whole project.**


> **Track: `pandas`** — the library used outside the classroom. A parallel `datascience`
> version of this notebook exists; `datascience_vs_pandas.md` maps one to the other. The
> **pandas note** callouts flag places where pandas behaves differently from `datascience`.


## Setup — run these first

The next cell defines small `check_*` helpers used after every question. They print `OK`
when an answer looks right and a short hint otherwise. They are guidance, not a grade, and
you don't need to read them.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(8)   # keep every run reproducible (Data 8, naturally)


In [ ]:
def _report(name, checks):
    for ok, hint in checks:
        if not ok:
            print('x  ' + name + ': ' + hint)
            return
    print('OK ' + name)


def check_written(name, text):
    bad = (not isinstance(text, str)) or len(text.strip()) < 15 or 'your answer' in text.lower()
    if bad:
        print('x  ' + name + ': write a sentence or two in the answer variable, replacing the placeholder')
    else:
        print('OK ' + name + '  (recorded - a person will read this one)')


def check_q0_1(n_rows, n_cols):
    _report('Q0.1', [
        (n_rows == 251, 'expected 251 rows, got ' + str(n_rows)),
        (n_cols == 6, 'expected 6 columns, got ' + str(n_cols)),
    ])


def check_q0_3(baseline_accuracy):
    ok = isinstance(baseline_accuracy, (int, float)) and 0.52 < baseline_accuracy < 0.54
    _report('Q0.3', [
        (ok, 'expected a fraction near 0.53 (133 of 251 rows), got ' + str(baseline_accuracy)),
    ])


def check_q1_1(fn):
    _report('Q1.1', [
        (fn('PQPQP', 'P') == 3, 'count_aa("PQPQP","P") should be 3, got ' + str(fn('PQPQP', 'P'))),
        (fn('PQPQP', 'Q') == 2, 'count_aa("PQPQP","Q") should be 2, got ' + str(fn('PQPQP', 'Q'))),
        (fn('AAAA', 'P') == 0, 'there is no P in "AAAA", so this should be 0'),
    ])


def check_q1_2(frac, pf, qf):
    try:
        got = [frac('PQPQ', 'P'), pf('PPPQ'), qf('PPPQ')]
    except Exception as e:
        print('x  Q1.2: a function raised an error - ' + str(e)); return
    if not all(isinstance(v, (int, float)) for v in got):
        print('x  Q1.2: each function should return a number (did you fill in the body?)'); return
    _report('Q1.2', [
        (abs(got[0] - 0.5) < 1e-9, 'aa_fraction("PQPQ","P") should be 0.5'),
        (abs(got[1] - 0.75) < 1e-9, 'proline_fraction("PPPQ") should be 0.75'),
        (abs(got[2] - 0.25) < 1e-9, 'glutamine_fraction("PPPQ") should be 0.25'),
    ])


def check_q1_4(longest_seq, shortest_seq, long_p_count, long_p_frac, short_p_count, short_p_frac):
    vals = [long_p_count, long_p_frac, short_p_count, short_p_frac]
    if not all(isinstance(v, (int, float)) for v in vals):
        print('x  Q1.4: set all four variables to numbers (use your Q1.1 / Q1.2 functions)'); return
    _report('Q1.4', [
        (long_p_count == longest_seq.count('P'), 'long_p_count should be the number of P in the longest sequence'),
        (short_p_count == shortest_seq.count('P'), 'short_p_count should be the number of P in the shortest sequence'),
        (abs(long_p_frac - longest_seq.count('P') / len(longest_seq)) < 1e-9, 'long_p_frac should be count / length'),
        (abs(short_p_frac - shortest_seq.count('P') / len(shortest_seq)) < 1e-9, 'short_p_frac should be count / length'),
    ])


def check_q0_2(per_grain):
    try:
        n = len(per_grain)
        top = per_grain.sort_values(ascending=False).index[0]
    except Exception:
        print('x  Q0.2: per_grain should be a Series of counts indexed by grain'); return
    _report('Q0.2', [
        (n == 10, 'expected 10 grains, got ' + str(n)),
        (top == 'wheat', 'the grain with the most proteins should be wheat, got ' + str(top)),
    ])


def check_q1_3(df):
    try:
        columns = set(df.columns)
    except Exception:
        print('x  Q1.3: run the column-adding code first'); return
    if not {'p_frac', 'q_frac', 'pq_frac'} <= columns:
        print('x  Q1.3: missing one of p_frac / q_frac / pq_frac'); return
    p, q, pq = df['p_frac'].to_numpy(), df['q_frac'].to_numpy(), df['pq_frac'].to_numpy()
    gy = df.loc[df['gluten'] == 'yes', 'pq_frac'].mean()
    gn = df.loc[df['gluten'] == 'no', 'pq_frac'].mean()
    _report('Q1.3', [
        (p.min() >= 0 and p.max() <= 1, 'p_frac must be between 0 and 1'),
        (np.allclose(pq, p + q), 'pq_frac should equal p_frac + q_frac'),
        (0.40 < gy < 0.55, 'mean pq_frac for gluten grains looks off (' + format(gy, '.3f') + ', expect ~0.47)'),
        (gy > gn + 0.2, 'gluten grains should have a much higher mean pq_frac than the rest'),
    ])


## Part 0 — Orientation  ·  5 points


Load the data into a `DataFrame` and look at the first few rows.


In [ ]:
grains = pd.read_csv('data/grains.csv')
grains.head(3)


### What each column means

| column | what it is | do we compute from it? |
|---|---|---|
| `seq_id` | our label for the row (`S001`, `S002`, ...) | no — it just lets us point at one specific protein |
| `grain` | the grain this protein comes from (`wheat`, `rice`, ...) | used for grouping, and for error analysis in Part 6 |
| `gluten` | `yes` if the grain is wheat, barley, or rye; otherwise `no` | **yes — this is the label we try to predict** |
| `protein_name` | the name the source database gives this protein | no calculations from it. In Part 6 it helps explain *which* proteins the model gets wrong. Ignore it until then. |
| `accession` | this protein's ID in **UniProt**, a public protein database | no — provenance only. Paste it into uniprot.org to see the original record. |
| `sequence` | the protein itself: a string of capital letters, one per amino acid | **yes — every number we build comes from this column** |

You'll see the same `protein_name` (and the same `grain`) on many rows. That's
expected: a grain has many storage-protein genes and many database entries, and
each row is one protein record.


### Question 0.1 · 2 pt

Set `n_rows` and `n_cols` to the number of rows and columns in `grains`.


In [ ]:
n_rows = grains.shape[0]
n_cols = grains.shape[1]

check_q0_1(n_rows, n_cols)


### Question 0.2 · 2 pt

Each row is one protein, so counting rows per grain tells you how many proteins we
have for each. Build `per_grain`: the count of proteins for every `grain`, largest
first.


In [ ]:
per_grain = grains['grain'].value_counts()
per_grain


> **pandas note.** `value_counts()` returns a `Series` indexed by grain, already
> sorted largest first. The `datascience` version returns a two-column `Table`
> (`grain`, `count`). Same information, different shape.


In [ ]:
check_q0_2(per_grain)


### Question 0.3 · 1 pt

Before building any model, find the score to beat. Suppose you ignored the
`sequence` entirely and, for every row, just guessed whichever `gluten` value shows
up more often. How often would that be right?

Count the `'no'` rows and the `'yes'` rows, then set `baseline_accuracy` to the
larger count divided by the total number of rows.


In [ ]:
n_no  = (grains['gluten'] == 'no').sum()
n_yes = (grains['gluten'] == 'yes').sum()
baseline_accuracy = max(n_no, n_yes) / len(grains)

print('always guess the majority label ->', baseline_accuracy)
check_q0_3(baseline_accuracy)


Now say why this number matters, in one or two sentences. (What would it mean if a
model you build only reached this accuracy?) Put your answer in `answer_q0_3`.


In [ ]:
answer_q0_3 = (
    'It is the accuracy of a model that ignores the sequence completely, so any real '
    'model has to beat it to be worth anything. Reaching only this number would mean '
    'the model learned the class sizes, not the biology.'
)

check_written('Q0.3 written', answer_q0_3)


## Part 1 — From a sequence to numbers  ·  12 points

A model can't work with the raw string `'MAAKIF...IF'`. It needs **features**:
numbers that summarize a sequence. In this part you build three of them —

* `p_frac`: the fraction of the protein that is proline (`P`)
* `q_frac`: the fraction that is glutamine (`Q`)
* `pq_frac`: the two added together

— and attach them to the table as new columns.

You'll start with a general helper, `count_aa(seq, aa)`. The name `aa` is short for
**amino acid**: it's just a parameter, so the one function works for any letter. In
Q1.2 you'll call it with `'P'` and `'Q'` specifically. We don't care about amino
acids in general — `aa` is only there so we don't have to write the counting logic
twice.


### Question 1.1 · 3 pt

Write `count_aa(seq, aa)` that returns how many times the single character `aa`
occurs in the string `seq`. A `for` loop over the letters of `seq` is a fine way to
do it.


In [ ]:
def count_aa(seq, aa):
    # number of times the character aa appears in the string seq
    aa_count = 0
    for letter in seq:
        if letter == aa:
            aa_count = aa_count + 1
    return aa_count

# (a one-line version, once the loop feels obvious: sum(letter == aa for letter in seq))
check_q1_1(count_aa)


### Question 1.2 · 3 pt

Write `aa_fraction(seq, aa)` = the fraction of `seq` that is the character `aa`
(a number between 0 and 1) — that's `count_aa` divided by the length of `seq`.

Then `proline_fraction(seq)` and `glutamine_fraction(seq)` are just `aa_fraction`
with the letter filled in: `'P'` for proline, `'Q'` for glutamine.


In [ ]:
def aa_fraction(seq, aa):
    return count_aa(seq, aa) / len(seq)

def proline_fraction(seq):
    return aa_fraction(seq, 'P')

def glutamine_fraction(seq):
    return aa_fraction(seq, 'Q')

check_q1_2(aa_fraction, proline_fraction, glutamine_fraction)


Here is the table again, so the column names are fresh before you add to it:


In [ ]:
grains.head(3)


### Question 1.3 · 4 pt

Add three columns to `grains`: `p_frac`, `q_frac`, and `pq_frac`
(= `p_frac + q_frac`), computed from the `sequence` column with the functions
you just wrote.

**Hint:** `grains['p_frac'] = grains['sequence'].apply(proline_fraction)` runs
the function down the column and stores the result. (No `grains = ...` needed
here — assigning into `grains['p_frac']` changes it in place.)


In [ ]:
grains['p_frac'] = grains['sequence'].apply(proline_fraction)
grains['q_frac'] = grains['sequence'].apply(glutamine_fraction)
grains['pq_frac'] = grains['p_frac'] + grains['q_frac']

grains[['seq_id', 'grain', 'gluten', 'p_frac', 'q_frac', 'pq_frac']].head(5)
check_q1_3(grains)


### Question 1.4 · 2 pt

Here's why we used a *fraction* and not a raw count. The two cells below pick out the
longest and the shortest protein in the dataset. For each, set the raw `P` count and
the proline fraction, using your functions from Q1.1 and Q1.2.


In [ ]:
longest_seq = max(grains['sequence'], key=len)
shortest_seq = min(grains['sequence'], key=len)
print('lengths:', len(longest_seq), 'vs', len(shortest_seq))


In [ ]:
long_p_count  = count_aa(longest_seq, 'P')
long_p_frac   = proline_fraction(longest_seq)
short_p_count = count_aa(shortest_seq, 'P')
short_p_frac  = proline_fraction(shortest_seq)

print('longest :', long_p_count, 'P out of', len(longest_seq), '->', long_p_frac)
print('shortest:', short_p_count, 'P out of', len(shortest_seq), '->', short_p_frac)
check_q1_4(longest_seq, shortest_seq, long_p_count, long_p_frac, short_p_count, short_p_frac)


You want to say which of these two proteins is *more proline-rich*. The raw counts
and the fractions can point to different answers. Which number answers that question
fairly, and why? One sentence in `answer_q1_4`.


In [ ]:
answer_q1_4 = (
    'The fraction, because it divides out length: a raw count is large for any long '
    'protein and small for any short one, so it mostly measures size, not how P-rich '
    'the protein is.'
)

check_written('Q1.4 written', answer_q1_4)


---
**Part 1 done.** `grains` now carries `p_frac`, `q_frac`, and `pq_frac`. Part 2
uses them to compare gluten vs non-gluten grains with a histogram and a scatter
plot — and to spot which "safe" grains look suspiciously gluten-like.
